In [1]:
from pathlib import Path
import sys

import torch

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
repo_root = next(path for path in candidates if (path / "src" / "qvnn_pc").exists())
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [ ]:
from qvnn_pc.models.pcn import PCNLayer, PredictiveCodingNetwork, train_pcn

torch.set_printoptions(precision=4, sci_mode=True)
torch.manual_seed(0)

In [7]:
pcn = PredictiveCodingNetwork(dims=[4, 20, 20, 20, 20], output_dim=2)

In [ ]:
from qvnn_pc.functions import ackley_4d, griewank_4d, rastrigin_4d, schwefel_4d
from qvnn_pc.data import make_benchmark_regression_split
from torch.utils.data import DataLoader, TensorDataset
FUNCTIONS = {
    "ackley": ackley_4d,
    "griewank": griewank_4d,
    "rastrigin": rastrigin_4d,
    "schwefel": schwefel_4d,
}

BOUNDS = {
    "ackley": (-30.0, 30.0),
    "griewank": (-600.0, 600.0),
    "rastrigin": (-5.12, 5.12),
    "schwefel": (-500.0, 500.0),
}


function_name = "ackley"
n = 30000
sample_seed = 1
split_seed = 2

split = make_benchmark_regression_split(
    function_name,
    n=n,
    sample_seed=sample_seed,
    split_seed=split_seed,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

x_train, y_train = split.x_train, split.y_train
x_val, y_val = split.x_val, split.y_val
x_test, y_test = split.x_test, split.y_test



In [9]:
# Create dataloaders
train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=500, shuffle=True)
val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=16)
test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=16)

In [ ]:
# Convert regression target to binary class labels for PCN training (LongTensor required by one_hot)
y_train_scalar = y_train[..., 0].squeeze(-1)
threshold = y_train_scalar.median()
y_train_cls = (y_train_scalar > threshold).long()

train_loader_cls = DataLoader(TensorDataset(x_train, y_train_cls), batch_size=16, shuffle=True)

train_pcn(
    model=pcn, 
    data_loader=train_loader_cls,
    num_epochs=100,
    eta_infer=0.05,
    eta_learn=0.005,
    T_infer=50,
    T_learn=500,
    device=x_train.device.type
)



  1%|          | 1/100 [06:10<10:11:28, 370.59s/it]

In [ ]:
train_pcn

<function qvnn_pc.models.pc.train_pcn(model, data_loader, num_epochs, eta_infer, eta_learn, T_infer, T_learn, device='cuda')>